# Milestone 1: HistWords data preparation

## Load historical embeddings

Read the HistWords files from 1800 to 1990.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

# HistWords folder
DATA_DIR = Path("/Users/nathanwu/Downloads/sgns").expanduser()
OUTPUT_DIR = Path("data/histwords")
YEARS = list(range(1800, 2000, 10))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.is_dir(), f"HistWords directory not found: {DATA_DIR}"
assert all((DATA_DIR / f"{y}-w.npy").exists() and
           (DATA_DIR / f"{y}-vocab.pkl").exists() for y in YEARS), "Some decade embedding/vocabulary files are missing"


def load_decade(year):

    with open(DATA_DIR / f"{year}-vocab.pkl", "rb") as file:
        vocab = pickle.load(file, encoding="latin1")
    vectors = np.load(DATA_DIR / f"{year}-w.npy", mmap_mode="r")
    if isinstance(vocab, dict):

        word_to_idx = vocab
    else:
        word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    assert len(word_to_idx) == vectors.shape[0], f"Mismatched vocabulary/vector rows for {year}"
    return vectors, word_to_idx


def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 and np.isfinite(denom) else np.nan

## Check whether decades are comparable

Compare a few words in 1940 and 1950. This is a quick check, not proof of alignment.

In [2]:
v1940, index1940 = load_decade(1940)
v1950, index1950 = load_decade(1950)
check_words = ["the", "and", "water", "house", "mother", "tree", "food", "dog",
               "cat", "book", "school", "city", "river", "family", "hand", "sun", "earth", "child"]
checks = []
for word in check_words:
    if word in index1940 and word in index1950:
        checks.append({"word": word, "cosine_similarity": cosine_similarity(
            v1940[index1940[word]], v1950[index1950[word]])})
alignment_check = pd.DataFrame(checks)
display(alignment_check)
print("Average similarity:", alignment_check["cosine_similarity"].mean())

,word,cosine_similarity
0,the,0.844231
1,and,0.394069
2,water,0.806045
3,house,0.757309
4,mother,0.818122
5,tree,0.764996
6,food,0.787765
7,dog,0.744779
8,cat,0.694497
9,book,0.796921


Average similarity: 0.7568075945055837


## Filter the vocabulary

Keep words listed in every decade and containing only English letters.

In [3]:
decade_data = {year: load_decade(year) for year in YEARS}
common_words = set.intersection(*(set(idx) for _, idx in decade_data.values()))
clean_words = sorted(word for word in common_words if word.isascii() and word.isalpha())
print("Words in every vocabulary:", len(common_words))
print("ASCII alphabetic candidates:", len(clean_words))
print("Examples:", clean_words[:20])

Words in every vocabulary: 100000
ASCII alphabetic candidates: 96470
Examples: ['a', 'aa', 'aaa', 'aaai', 'aaas', 'aaay', 'aachen', 'aaf', 'aan', 'aar', 'aargau', 'aarhus', 'aaron', 'aarp', 'aas', 'aass', 'ab', 'aba', 'ababa', 'abaca']


## Calculate changes between decades

Use cosine distance for each word in consecutive decades. Zero vectors produce missing values.

In [4]:
records = []
for start_year, end_year in zip(YEARS[:-1], YEARS[1:]):
    vec1, idx1 = decade_data[start_year]
    vec2, idx2 = decade_data[end_year]
    for word in clean_words:
        sim = cosine_similarity(vec1[idx1[word]], vec2[idx2[word]])
        records.append({
            "word": word,
            "start_year": start_year,
            "end_year": end_year,
            "semantic_change": 1 - sim if np.isfinite(sim) else np.nan,
        })
changes_df = pd.DataFrame.from_records(records)
print("Total measurements:", len(changes_df))
display(changes_df.head())

Total measurements: 1832930


,word,start_year,end_year,semantic_change
0,a,1800,1810,0.439643
1,aa,1800,1810,NaN
2,aaa,1800,1810,NaN
3,aaai,1800,1810,NaN
4,aaas,1800,1810,NaN


## Remove words with missing values

Keep words with valid change scores across all the selected decades.

In [5]:
bad = ~np.isfinite(changes_df["semantic_change"])
print(f"Invalid scores: {bad.sum():,} ({bad.mean():.1%})")
print("Invalid scores by starting decade:")
display(changes_df.assign(invalid=bad).groupby("start_year")["invalid"].sum())

invalid_words = set(changes_df.loc[bad, "word"])
valid_words = sorted(set(clean_words) - invalid_words)
changes_clean = changes_df.loc[changes_df["word"].isin(valid_words)].copy()
assert np.isfinite(changes_clean["semantic_change"]).all()
print("Words with complete trajectories:", len(valid_words))
print("Usable interval measurements:", len(changes_clean))

Invalid scores: 1,222,691 (66.7%)
Invalid scores by starting decade:


start_year
1800    83636
1810    81031
1820    77580
1830    75772
1840    73460
1850    70847
1860    70693
1870    68361
1880    64089
1890    60614
1900    62227
1910    63498
1920    64754
1930    65419
1940    63707
1950    56480
1960    45650
1970    40009
1980    34864
Name: invalid, dtype: int64

Words with complete trajectories: 12599
Usable interval measurements: 239381


## Organize changes by word

Each row contains a word and its observed changes across decades.

In [6]:
trajectory_df = changes_clean.pivot(
    index="word", columns="start_year", values="semantic_change"
).sort_index(axis=1)
trajectory_df.columns = [f"{year}-{year + 10}" for year in trajectory_df.columns]
assert trajectory_df.shape == (len(valid_words), len(YEARS) - 1)
display(trajectory_df.head())
print("Trajectory shape:", trajectory_df.shape)

,1800-1810,1810-1820,1820-1830,1830-1840,1840-1850,1850-1860,1860-1870,1870-1880,1880-1890,1890-1900,1900-1910,1910-1920,1920-1930,1930-1940,1940-1950,1950-1960,1960-1970,1970-1980,1980-1990
word,,,,,,,,,,,,,,,,,,,
a,0.439643,0.455142,0.529408,0.519967,0.472808,0.444678,0.442922,0.398035,0.460105,0.489015,0.437273,0.442004,0.485621,0.425161,0.429265,0.458286,0.466207,0.468693,0.435592
aaron,0.341612,0.396481,0.373028,0.399939,0.370898,0.444521,0.423422,0.359942,0.398053,0.425841,0.452328,0.455419,0.474083,0.483544,0.453653,0.429581,0.523840,0.477133,0.467052
ab,0.346193,0.294959,0.317876,0.344740,0.306875,0.395935,0.308898,0.277492,0.311529,0.292530,0.366389,0.339946,0.379917,0.350476,0.388804,0.388188,0.446668,0.406596,0.439291
abandon,0.235984,0.270186,0.266253,0.286864,0.267079,0.264191,0.264422,0.269184,0.322742,0.236623,0.243800,0.233368,0.310850,0.301121,0.269931,0.219098,0.281847,0.297363,0.255649
abandoned,0.261701,0.282700,0.291175,0.309909,0.265875,0.259465,0.255481,0.253382,0.252037,0.261412,0.222838,0.267185,0.249312,0.261601,0.221069,0.227618,0.242495,0.242323,0.243953


Trajectory shape: (12599, 19)


## Organize historical changes for modeling

Put each decade’s observed change beside the preceding decade’s change. This only prepares inputs and targets; no model is trained or run.

In [7]:
intervals = list(trajectory_df.columns)
records = []
for word, row in trajectory_df.iterrows():
    values = row.to_numpy(dtype=float)
    for i in range(1, len(intervals)):
        start, end = map(int, intervals[i].split("-"))
        records.append({
            "word": word,
            "start_year": start,
            "end_year": end,
            "previous_change": float(values[i - 1]),
            "actual_change": float(values[i]),
        })
forecast_df = pd.DataFrame(records)
assert not forecast_df.isna().any().any()
print("Forecast examples:", len(forecast_df))
display(forecast_df.head())

Forecast examples: 226782


,word,start_year,end_year,previous_change,actual_change
0,a,1810,1820,0.439643,0.455142
1,a,1820,1830,0.455142,0.529408
2,a,1830,1840,0.529408,0.519967
3,a,1840,1850,0.519967,0.472808
4,a,1850,1860,0.472808,0.444678


## Split the data by time

Use target intervals ending by 1960 for training, 1960–1970 for validation, and 1970–1990 for testing.

In [8]:
train_df = forecast_df.loc[forecast_df.end_year <= 1960].copy()
dev_df = forecast_df.loc[forecast_df.end_year == 1970].copy()
test_df = forecast_df.loc[forecast_df.end_year >= 1980].copy()

assert len(train_df) + len(dev_df) + len(test_df) == len(forecast_df)
assert set(train_df.end_year).isdisjoint(dev_df.end_year)
assert set(test_df.end_year).isdisjoint(dev_df.end_year)
assert set(train_df.end_year).isdisjoint(test_df.end_year)

for name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    print(f"{name}: {len(df):,} examples; target end years {sorted(df.end_year.unique())}")
    assert np.isfinite(df[["previous_change", "actual_change"]].to_numpy()).all()

display(train_df.head())

train: 188,985 examples; target end years [np.int64(1820), np.int64(1830), np.int64(1840), np.int64(1850), np.int64(1860), np.int64(1870), np.int64(1880), np.int64(1890), np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960)]
dev: 12,599 examples; target end years [np.int64(1970)]
test: 25,198 examples; target end years [np.int64(1980), np.int64(1990)]


,word,start_year,end_year,previous_change,actual_change
0,a,1810,1820,0.439643,0.455142
1,a,1820,1830,0.455142,0.529408
2,a,1830,1840,0.529408,0.519967
3,a,1840,1850,0.519967,0.472808
4,a,1850,1860,0.472808,0.444678


## Save the datasets

In [9]:
for name, frame in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
changes_clean.to_csv(OUTPUT_DIR / "observed_changes.csv", index=False)
trajectory_df.to_csv(OUTPUT_DIR / "semantic_trajectories.csv")
print("Saved:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", path)

Saved:
 - data/histwords/dev.csv
 - data/histwords/observed_changes.csv
 - data/histwords/semantic_trajectories.csv
 - data/histwords/test.csv
 - data/histwords/train.csv
